In [1]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path("../data/raw/fuelhh")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

files = sorted(RAW_DIR.glob("fuelhh_*.csv"))

dfs = []

for file in files:
    print(f"Loading {file.name}...")
    
    df_year = pd.read_csv(
        file,
        dtype={
            "dataset": "string",
            "fuelType": "string"
        }
    )
    
    dfs.append(df_year)

fuelhh = pd.concat(
    dfs,
    ignore_index=True
)

print(f"\nTotal rows: {len(fuelhh):,}")
print(f"Columns: {fuelhh.columns.tolist()}")

Loading fuelhh_2020.csv...
Loading fuelhh_2021.csv...
Loading fuelhh_2022.csv...
Loading fuelhh_2023.csv...
Loading fuelhh_2024.csv...
Loading fuelhh_2025.csv...

Total rows: 1,903,057
Columns: ['dataset', 'publishTime', 'startTime', 'settlementDate', 'settlementPeriod', 'fuelType', 'generation']


In [2]:
fuelhh["startTime"] = pd.to_datetime(
    fuelhh["startTime"],
    utc=True
)

fuelhh["publishTime"] = pd.to_datetime(
    fuelhh["publishTime"],
    utc=True
)

In [3]:
fuelhh["datetime_uk"] = (
    fuelhh["startTime"]
    .dt.tz_convert("Europe/London")
)

In [4]:
duplicate_key_count = fuelhh.duplicated(
    subset=["startTime", "fuelType"]
).sum()

print(
    f"Duplicate startTime + fuelType rows: "
    f"{duplicate_key_count:,}"
)

Duplicate startTime + fuelType rows: 0


In [5]:
fuel_profile = (
    fuelhh
    .groupby("fuelType")
    .agg(
        records=("fuelType", "size"),
        min_generation_mw=("generation", "min"),
        max_generation_mw=("generation", "max"),
        mean_generation_mw=("generation", "mean"),
    )
    .sort_index()
)

fuel_profile

,records,min_generation_mw,max_generation_mw,mean_generation_mw
fuelType,,,,
BIOMASS,105170,0,3368,1977.439954
CCGT,105170,0,27339,10447.920671
COAL,105170,0,4392,341.140772
INTELE,9,0,0,0.000000
INTELEC,75310,-1034,1260,314.040632
INTEW,105170,-588,506,-95.569288
INTFR,105170,-2072,2630,787.154702
INTGRNL,24622,-516,506,-228.434327
INTIFA2,102224,-1626,1398,262.422288


In [6]:
sorted(fuelhh["fuelType"].dropna().unique())

['BIOMASS',
 'CCGT',
 'COAL',
 'INTELE',
 'INTELEC',
 'INTEW',
 'INTFR',
 'INTGRNL',
 'INTIFA2',
 'INTIRL',
 'INTNED',
 'INTNEM',
 'INTNSL',
 'INTVKL',
 'NPSHYD',
 'NUCLEAR',
 'OCGT',
 'OIL',
 'OTHER',
 'PS',
 'WIND']

In [7]:
timestamps = (
    fuelhh["startTime"]
    .drop_duplicates()
    .sort_values()
)

gaps = timestamps.diff().dropna()

gap_summary = (
    gaps
    .value_counts()
    .sort_index()
)

gap_summary

startTime
0 days 00:30:00    105130
0 days 01:00:00        33
0 days 01:30:00         4
0 days 02:00:00         2
Name: count, dtype: int64

In [8]:
print(
    gap_summary[
        gap_summary.index != pd.Timedelta("0 days 00:30:00")
    ]
)

startTime
0 days 01:00:00    33
0 days 01:30:00     4
0 days 02:00:00     2
Name: count, dtype: int64


In [9]:
fuelhh["date"] = fuelhh["datetime_uk"].dt.date
fuelhh["hour"] = fuelhh["datetime_uk"].dt.hour
fuelhh["minute"] = fuelhh["datetime_uk"].dt.minute
fuelhh["day_of_week"] = fuelhh["datetime_uk"].dt.day_name()
fuelhh["month"] = fuelhh["datetime_uk"].dt.month
fuelhh["year"] = fuelhh["datetime_uk"].dt.year
fuelhh["is_weekend"] = (
    fuelhh["datetime_uk"].dt.dayofweek >= 5
)

In [10]:
generation_categories = {
    "BIOMASS": "Biomass",
    "CCGT": "Gas",
    "COAL": "Coal",
    "NPSHYD": "Hydro",
    "NUCLEAR": "Nuclear",
    "OCGT": "Gas",
    "OIL": "Oil",
    "OTHER": "Other",
    "PS": "Pumped Storage",
    "WIND": "Wind",
}

interconnector_categories = {
    "INTELEC": "ElecLink",
    "INTEW": "East-West",
    "INTFR": "France IFA",
    "INTGRNL": "Greenlink",
    "INTIFA2": "France IFA2",
    "INTIRL": "Moyle",
    "INTNED": "Netherlands BritNed",
    "INTNEM": "Belgium Nemo",
    "INTNSL": "Norway North Sea Link",
    "INTVKL": "Denmark Viking Link",
}

In [11]:
def classify_fuel(fuel):
    if fuel in generation_categories:
        return "Generation"
    if fuel in interconnector_categories:
        return "Interconnector"
    return "Unclassified"

In [12]:
fuelhh["data_type"] = fuelhh["fuelType"].apply(classify_fuel)

In [13]:
def classify_category(fuel):
    if fuel in generation_categories:
        return generation_categories[fuel]
    if fuel in interconnector_categories:
        return interconnector_categories[fuel]
    return "Unclassified"

fuelhh["analysis_category"] = (
    fuelhh["fuelType"]
    .apply(classify_category)
)

In [14]:
fuelhh[
    ["fuelType", "data_type", "analysis_category"]
].drop_duplicates().sort_values(
    ["data_type", "fuelType"]
)

,fuelType,data_type,analysis_category
0,BIOMASS,Generation,Biomass
1,CCGT,Generation,Gas
2,COAL,Generation,Coal
8,NPSHYD,Generation,Hydro
9,NUCLEAR,Generation,Nuclear
10,OCGT,Generation,Gas
11,OIL,Generation,Oil
12,OTHER,Generation,Other
13,PS,Generation,Pumped Storage
14,WIND,Generation,Wind


In [15]:
print(
    fuelhh["data_type"].value_counts()
)

data_type
Generation        1051700
Interconnector     851348
Unclassified            9
Name: count, dtype: int64


In [16]:
fuelhh["date"] = fuelhh["datetime_uk"].dt.date
fuelhh["hour"] = fuelhh["datetime_uk"].dt.hour
fuelhh["minute"] = fuelhh["datetime_uk"].dt.minute

fuelhh["day_of_week"] = (
    fuelhh["datetime_uk"].dt.day_name()
)

fuelhh["month"] = (
    fuelhh["datetime_uk"].dt.month
)

fuelhh["year"] = (
    fuelhh["datetime_uk"].dt.year
)

fuelhh["is_weekend"] = (
    fuelhh["datetime_uk"].dt.dayofweek >= 5
)

In [17]:
processed_columns = [
    "startTime",
    "datetime_uk",
    "settlementDate",
    "settlementPeriod",
    "fuelType",
    "data_type",
    "analysis_category",
    "generation",
    "date",
    "hour",
    "minute",
    "day_of_week",
    "month",
    "year",
    "is_weekend",
]

fuelhh_clean = fuelhh[processed_columns].copy()

In [18]:
fuelhh_clean.head()

,startTime,datetime_uk,settlementDate,settlementPeriod,fuelType,data_type,analysis_category,generation,date,hour,minute,day_of_week,month,year,is_weekend
0,2020-01-07 23:00:00+00:00,2020-01-07 23:00:00+00:00,2020-01-07,47,BIOMASS,Generation,Biomass,1885,2020-01-07,23,0,Tuesday,1,2020,False
1,2020-01-07 23:00:00+00:00,2020-01-07 23:00:00+00:00,2020-01-07,47,CCGT,Generation,Gas,3875,2020-01-07,23,0,Tuesday,1,2020,False
2,2020-01-07 23:00:00+00:00,2020-01-07 23:00:00+00:00,2020-01-07,47,COAL,Generation,Coal,568,2020-01-07,23,0,Tuesday,1,2020,False
3,2020-01-07 23:00:00+00:00,2020-01-07 23:00:00+00:00,2020-01-07,47,INTEW,Interconnector,East-West,0,2020-01-07,23,0,Tuesday,1,2020,False
4,2020-01-07 23:00:00+00:00,2020-01-07 23:00:00+00:00,2020-01-07,47,INTFR,Interconnector,France IFA,-666,2020-01-07,23,0,Tuesday,1,2020,False


In [19]:
print(f"Rows: {len(fuelhh_clean):,}")
print(
    "Duplicate analytical keys:",
    fuelhh_clean.duplicated(
        subset=["startTime", "fuelType"]
    ).sum()
)
print(
    "Missing values:",
    fuelhh_clean.isna().sum().sum()
)

Rows: 1,903,057
Duplicate analytical keys: 0
Missing values: 0


In [20]:
output_path = (
    PROCESSED_DIR / "fuelhh_clean.parquet"
)

fuelhh_clean.to_parquet(
    output_path,
    index=False
)

print(f"Saved to: {output_path}")

Saved to: ..\data\processed\fuelhh_clean.parquet


In [21]:
fuelhh[
    ["fuelType", "data_type", "analysis_category"]
].drop_duplicates().sort_values(
    ["data_type", "fuelType"]
)

,fuelType,data_type,analysis_category
0,BIOMASS,Generation,Biomass
1,CCGT,Generation,Gas
2,COAL,Generation,Coal
8,NPSHYD,Generation,Hydro
9,NUCLEAR,Generation,Nuclear
10,OCGT,Generation,Gas
11,OIL,Generation,Oil
12,OTHER,Generation,Other
13,PS,Generation,Pumped Storage
14,WIND,Generation,Wind


In [22]:
fuelhh["data_type"].value_counts()

data_type
Generation        1051700
Interconnector     851348
Unclassified            9
Name: count, dtype: int64

In [23]:
print(f"Rows: {len(fuelhh_clean):,}")

print(
    "Duplicate analytical keys:",
    fuelhh_clean.duplicated(
        subset=["startTime", "fuelType"]
    ).sum()
)

print(
    "Missing values:",
    fuelhh_clean.isna().sum().sum()
)

Rows: 1,903,057
Duplicate analytical keys: 0
Missing values: 0


In [24]:
output_path = PROCESSED_DIR / "fuelhh_clean.parquet"

fuelhh_clean.to_parquet(
    output_path,
    index=False
)

print(f"Saved to: {output_path}")

Saved to: ..\data\processed\fuelhh_clean.parquet


In [25]:
test = pd.read_parquet(
    "../data/processed/fuelhh_clean.parquet"
)

print(f"Rows: {len(test):,}")
print(f"Columns: {test.columns.tolist()}")
print(
    "Duplicate keys:",
    test.duplicated(
        subset=["startTime", "fuelType"]
    ).sum()
)

Rows: 1,903,057
Columns: ['startTime', 'datetime_uk', 'settlementDate', 'settlementPeriod', 'fuelType', 'data_type', 'analysis_category', 'generation', 'date', 'hour', 'minute', 'day_of_week', 'month', 'year', 'is_weekend']
Duplicate keys: 0


In [26]:
print(f"Rows: {len(test):,}")
print(f"Columns: {test.columns.tolist()}")

Rows: 1,903,057
Columns: ['startTime', 'datetime_uk', 'settlementDate', 'settlementPeriod', 'fuelType', 'data_type', 'analysis_category', 'generation', 'date', 'hour', 'minute', 'day_of_week', 'month', 'year', 'is_weekend']
